# Week 4: Model Evaluation and Validation
**Bank Marketing Dataset | Machine Learning Engineer Internship**

This notebook rigorously evaluates the Logistic Regression baseline from Week 3, using
stratified k-fold cross-validation, multiple evaluation metrics, and error analysis to
understand not just *how well* the model performs but *where and why* it fails.

## 1. Rebuild Pipeline and Model (from Week 2–3)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score, classification_report)

df = pd.read_csv("bank-additional.csv", sep=";")

df['contacted_before'] = (df['pdays'] != 999).astype(int)
df['pdays_clean'] = df['pdays'].replace(999, 0)
df = df.drop(columns=['pdays'])
df_deploy = df.drop(columns=['duration'])

nominal_cols = ['job', 'marital', 'contact', 'poutcome', 'month', 'day_of_week']
df_encoded = pd.get_dummies(df_deploy, columns=nominal_cols)

edu_order = ['unknown', 'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
             'high.school', 'professional.course', 'university.degree']
df_encoded['education'] = df_deploy['education'].map({v: i for i, v in enumerate(edu_order)})

for col in ['default', 'housing', 'loan']:
    df_encoded[col] = df_deploy[col].map({'no': 0, 'yes': 1, 'unknown': -1})
df_encoded['y'] = df_encoded['y'].map({'no': 0, 'yes': 1})

numeric_cols = ['age', 'campaign', 'previous', 'pdays_clean',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

X = df_encoded.drop(columns=['y'])
y = df_encoded['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
print("Model retrained. Train shape:", X_train.shape, "Test shape:", X_test.shape)

Model retrained. Train shape: (3295, 50) Test shape: (824, 50)


## 2. Stratified K-Fold Cross-Validation

Why stratified k-fold rather than a single train-test split: a single split gives one estimate
of performance, which could be optimistic or pessimistic purely by chance depending on which
rows landed in the test set — especially risky with an imbalanced target like this one. Stratified
k-fold repeats the evaluation across 5 different folds, **each preserving the original ~89/11
class ratio**, giving both an average performance estimate and a sense of how much that estimate
varies (its standard deviation) — i.e. how much to trust it.

In [2]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(model, X_train, y_train, cv=skf,
                             scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])

print(f"{'Metric':<10}{'Mean':>8}{'Std':>8}")
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    scores = cv_results[f'test_{metric}']
    print(f"{metric:<10}{scores.mean():>8.3f}{scores.std():>8.3f}")

Metric        Mean     Std
accuracy     0.820   0.012
precision    0.328   0.024
recall       0.615   0.073
f1           0.427   0.035
roc_auc      0.768   0.037


**Reading the spread:** recall varies the most across folds (mean 0.615, std 0.073) — meaning
which specific clients land in a given fold noticeably affects how many true subscribers get
caught. This is expected given how few positive examples exist per fold (roughly 90 per fold),
and is itself a useful finding: it tells us this model's recall estimate carries real uncertainty,
not just its point value.

## 3. Held-Out Test Set Evaluation

In [3]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy: ", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:   ", round(recall_score(y_test, y_pred), 3))
print("F1 Score: ", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba), 3))

print("\n" + classification_report(y_test, y_pred, target_names=['no', 'yes']))

Accuracy:  0.846
Precision: 0.372
Recall:    0.6
F1 Score:  0.46
ROC-AUC:   0.772

              precision    recall  f1-score   support

          no       0.95      0.88      0.91       734
         yes       0.37      0.60      0.46        90

    accuracy                           0.85       824
   macro avg       0.66      0.74      0.68       824
weighted avg       0.88      0.85      0.86       824



**Cross-validation vs. held-out test set comparison:** CV recall mean was 0.615; the held-out
test recall is 0.600 — very close. This closeness is a good sign: it means the model is not
overfit to a lucky train-test split, and its performance should generalize reasonably to new,
unseen clients. If the test score had been dramatically different from the CV mean, that would
suggest the single split was unusually easy or hard, and shouldn't be trusted alone.

## 4. Confusion Matrix and Overfitting/Underfitting Check

In [4]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives:  {tn}  (correctly predicted 'no')")
print(f"False Positives: {fp}  (predicted 'yes', actually 'no')")
print(f"False Negatives: {fn}  (predicted 'no', actually 'yes' — missed subscribers)")
print(f"True Positives:  {tp}  (correctly predicted 'yes')")

Confusion Matrix:
[[643  91]
 [ 36  54]]

True Negatives:  643  (correctly predicted 'no')
False Positives: 91  (predicted 'yes', actually 'no')
False Negatives: 36  (predicted 'no', actually 'yes' — missed subscribers)
True Positives:  54  (correctly predicted 'yes')


In [5]:
# Overfitting check: compare training performance to test performance
train_pred = model.predict(X_train)
print("Training accuracy:", round(accuracy_score(y_train, train_pred), 3))
print("Test accuracy:    ", round(accuracy_score(y_test, y_pred), 3))
print("Training recall:  ", round(recall_score(y_train, train_pred), 3))
print("Test recall:      ", round(recall_score(y_test, y_pred), 3))

Training accuracy: 0.831
Test accuracy:     0.846
Training recall:   0.629
Test recall:       0.6


Training and test scores are close to each other (and close to the cross-validation mean),
with no large gap in either direction. A large train-score-minus-test-score gap would signal
**overfitting** (the model memorizing training data rather than learning generalizable patterns);
uniformly poor scores on both would signal **underfitting** (the model too simple to capture the
real relationship). Neither is happening here — Logistic Regression's linear simplicity is
actually working in its favor for this dataset size, at the cost of a hard ceiling on how much
more accurate it could get without added complexity.

## 5. Error Analysis

Aggregate metrics say *how often* the model is wrong. This section looks at *which* clients it
gets wrong, to find out if the errors follow a pattern.

In [6]:
orig_test = df_deploy.loc[X_test.index].copy()
orig_test['actual'] = y_test.values
orig_test['predicted'] = y_pred

false_negatives = orig_test[(orig_test['actual'] == 1) & (orig_test['predicted'] == 0)]
false_positives = orig_test[(orig_test['actual'] == 0) & (orig_test['predicted'] == 1)]
true_positives  = orig_test[(orig_test['actual'] == 1) & (orig_test['predicted'] == 1)]
true_negatives  = orig_test[(orig_test['actual'] == 0) & (orig_test['predicted'] == 0)]

print("False negatives (missed real subscribers):", len(false_negatives))
print("False positives (false alarms):", len(false_positives))

False negatives (missed real subscribers): 36
False positives (false alarms): 91


In [7]:
# Does prior campaign outcome (poutcome) explain missed subscribers?
print("poutcome among FALSE NEGATIVES (missed):")
print(false_negatives['poutcome'].value_counts(normalize=True).round(2))
print("\npoutcome among TRUE POSITIVES (correctly caught):")
print(true_positives['poutcome'].value_counts(normalize=True).round(2))

poutcome among FALSE NEGATIVES (missed):
poutcome
nonexistent    0.89
failure        0.11
Name: proportion, dtype: float64

poutcome among TRUE POSITIVES (correctly caught):
poutcome
nonexistent    0.59
success        0.31
failure        0.09
Name: proportion, dtype: float64


**Finding:** 89% of missed subscribers (false negatives) had `poutcome = 'nonexistent'`
(no prior campaign contact), compared to only 59% of correctly caught subscribers. In other
words, the model leans heavily on a client's *prior campaign history* as a signal — when a
client has a prior "success" on record, the model catches them reliably (31% of true positives
had `poutcome = 'success'`), but for genuinely new prospects with no track record, the model
has a much weaker basis to work from and is more likely to miss a real "yes."

This is a real limitation, not just a metric — it means the model is currently better at
re-identifying past responders than discovering new ones, which matters for how a bank would
actually use it.

In [8]:
# Does contact frequency explain false alarms?
print("Mean 'campaign' (contacts this campaign) — False Positives vs True Negatives:")
print("False positives:", round(false_positives['campaign'].mean(), 2))
print("True negatives: ", round(true_negatives['campaign'].mean(), 2))

Mean 'campaign' (contacts this campaign) — False Positives vs True Negatives:
False positives: 1.81
True negatives:  2.82


**Finding:** false positives average 1.81 contacts versus 2.82 for correctly identified
non-subscribers. This suggests the model tends to over-predict "yes" for clients contacted only
once or twice — plausibly because low contact count correlates with *early-stage* campaigns
where other features (like season or economic indicators) look favorable, even though this
particular client ultimately said no.

## 6. Model Interpretability: What Drives the Predictions

In [9]:
coefs = pd.Series(model.coef_[0], index=X_train.columns).sort_values()
print("Strongest push toward 'NO':")
print(coefs.head(5).round(3))
print("\nStrongest push toward 'YES':")
print(coefs.tail(5).round(3))

Strongest push toward 'NO':
emp.var.rate        -1.315
job_unknown         -1.314
month_nov           -0.705
poutcome_failure    -0.608
job_self-employed   -0.544
dtype: float64

Strongest push toward 'YES':
marital_unknown     0.457
cons.price.idx      0.641
month_dec           0.767
contacted_before    0.826
month_mar           1.283
dtype: float64


`contacted_before` and `month_mar` are among the strongest positive drivers — clients
previously contacted, and campaigns run in March specifically, are associated with a higher
subscription likelihood (a seasonal effect documented in the original UCI study, plausibly tied
to that period's tax or financial planning). `emp.var.rate` (employment variation rate) is the
single strongest negative driver — when the broader economy is tightening, clients are less
inclined to lock money into a term deposit. This aligns with financial intuition rather than
looking like a modeling artifact, which is a good interpretability sanity check.

## 7. Decision Threshold Sensitivity

The default 0.5 threshold is a convention, not a requirement. Sweeping across thresholds makes
the precision/recall trade-off concrete rather than theoretical.

In [10]:
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (y_proba >= t).astype(int)
    p = precision_score(y_test, pred)
    r = recall_score(y_test, pred)
    f = f1_score(y_test, pred)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"threshold={t}  precision={p:.3f}  recall={r:.3f}  f1={f:.3f}  TP={tp}  FP={fp}  FN={fn}")

threshold=0.3  precision=0.154  recall=0.833  f1=0.260  TP=75  FP=413  FN=15
threshold=0.4  precision=0.219  recall=0.667  f1=0.330  TP=60  FP=214  FN=30
threshold=0.5  precision=0.372  recall=0.600  f1=0.460  TP=54  FP=91  FN=36
threshold=0.6  precision=0.446  recall=0.500  f1=0.471  TP=45  FP=56  FN=45
threshold=0.7  precision=0.519  recall=0.456  f1=0.485  TP=41  FP=38  FN=49


Lowering the threshold to 0.3 catches 75 of 90 true subscribers (recall 0.833) at the cost
of 413 false alarms. Raising it to 0.7 improves precision to 0.519 but recall falls to 0.456. The
"right" threshold depends on a real business cost trade-off (agent time vs. missed deposit value)
that isn't established in this analysis — an honest limitation rather than a default picked
arbitrarily.

## 8. Limitations

1. **Linearity assumption**: Logistic Regression cannot capture non-linear effects (e.g. a
   possible non-linear relationship between age and subscription likelihood) without explicit
   feature engineering.
2. **Feature ceiling**: a false negative and false positive example (Section 5) were nearly
   identical on every available feature yet had opposite true outcomes — evidence of a real
   limit on how separable the classes are with this feature set alone.
3. **Small positive-class sample**: ~90 positive test examples means single-record conclusions
   and the recall estimate itself carry real uncertainty (reflected in the CV fold std of 0.073).
4. **Single train-test split for final reporting**: nested cross-validation would be more rigorous.
5. **Temporal validity untested**: this is historical campaign data; concept drift over time is
   unverified.
6. **Threshold not cost-optimized**: 0.5 is reasonable but arbitrary without a real cost ratio.

## 9. Summary

- **Cross-validation confirms stability**: recall (mean 0.615, std 0.073) and ROC-AUC (mean 0.768)
  are consistent across folds, and the held-out test set closely matches the CV mean —
  no meaningful sign of overfitting.
- **Error analysis reveals a specific limitation**: the model relies heavily on prior campaign
  history (`poutcome`) and under-performs on clients with no prior contact record.
- **Coefficients are interpretable and intuitive**: contact history and seasonality push toward
  "yes"; a tightening economy pushes toward "no" — consistent with real-world financial behavior.
- These findings directly set up Week 5: hyperparameter tuning will be evaluated against this
  exact baseline (same seed, same CV strategy), and the `poutcome` reliance issue is a candidate
  for targeted feature engineering in a future iteration.